# 05 · Check the results: the final file (input_flytxt) and the scored 360 view, rule by rule

Run it after `04_agents_nba_fast_ollama_real25k` (or 03). It **only reads** the run's files and shows every check as a
table (a DataFrame you can filter):

| Part | What is checked |
|---|---|
| 1 | what was scored: target lines, lines in the file, why the others were dropped |
| 2 | the file itself: 38 columns, one row per MSISDN, dates, 6 offers per line |
| 3 | every offer: in the catalogue, both scripts, the opportunity name |
| 4 | the code-5 rules: no repeated offer (F-11), no two offers of the same type and price (F-09), no conflict pair (F-12) |
| 5 | daily / weekly / monthly: the slots (1, 2, 5 = 30 days; 3, 4, 6 = 7 days) and the mix per line |
| 6 | why each offer is there: recommended, favourite, forced bundle, top-up; and the favourites really bought |
| 7 | **same price, different allocation** (e.g. 5 R for 15 GB and 5 R for 20 GB in one line) and **dominated offers** (the same money or more for less) |
| 8 | the price order of the slots and the price against the customer's usual spend (cannibalization) |
| 9 | the actions: per segment, upsell to an at-risk segment (H-06), upsell with churn >= 0.60, the action's offer families |
| 10 | the 360 view against the final file, the most recommended offers |
| 11 | **the scorecard**: every check, its value, what is expected, OK / CHECK |

**No MSISDN is printed.** The examples show `LINE`, the row number in the final file. The files hold MSISDNs: keep
this notebook's outputs on this laptop, do not open the files in Excel.

In [ ]:
# --- settings -------------------------------------------------------------------
RUN_ROOT = "data/real25k"     # the data root of the run: "data/real25k" (notebook 04 real) | "data" (notebook 03)
RUN_DATE = None               # None = the latest run in <RUN_ROOT>/output/; or "2026-11-17"
TARGET = "target2"
EXAMPLES = 10                 # rows shown for each example table

In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_columns", 60, "display.width", 220, "display.max_colwidth", 80, "display.max_rows", 200)
ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
BASE = ROOT / RUN_ROOT
_runs = sorted(p for p in (BASE / "output").glob("20??-??-??") if (p / TARGET).is_dir())
if not _runs and not RUN_DATE:
    raise SystemExit(f"no run in {BASE / 'output'}: set RUN_ROOT (\"data/real25k\" or \"data\")")
OUT = BASE / "output" / RUN_DATE if RUN_DATE else _runs[-1]
D = OUT.name
WEEK = next((p for p in (BASE / "weekly" / D, ROOT / "data" / "weekly" / D) if p.is_dir()), None)
if WEEK is None:
    raise SystemExit(f"no week folder for {D} (the catalogue, scripts and conflict pairs are read there)")
CHECKS = []
SLOTS = range(1, 7)
MONTHLY_SLOTS, WEEKLY_SLOTS = (1, 2, 5), (3, 4, 6)
ACTION_FAMILIES = {"MAINTAIN": ("COMBO", "DATA"), "UPSELL": ("COMBO", "DATA"), "RETAIN": ("COMBO", "DATA"),
                   "UPSELL VOICE": ("DATA", "COMBO"), "CROSS SELL VOICE": ("DATA", "COMBO"),
                   "CROSS SELL DATA": ("VOICE", "COMBO"), "UPSELL DATA": ("VOICE", "COMBO")}
UPSELL_ACTIONS = {"UPSELL", "UPSELL DATA", "UPSELL VOICE"}
AT_RISK_SEGMENTS = {"Nursering_Early_Churn_At_Risk", "Onboarding_Early_Churn_At_Risk", "Retention_RGS_At_Risk"}
H06_EXEMPT = {"Onboarding_Early_Churn_At_Risk": "UPSELL"}   # its production action (S-3) is UPSELL: allowed


def rd(p):
    return pd.read_csv(p, dtype=str, keep_default_na=False, encoding="utf-8-sig")


def table(name):
    p = next(WEEK.rglob(f"{name}.csv.gz"), None) or next(WEEK.rglob(f"{name}.csv"), None)
    if p is None:
        print(f"note: {name} not found in {WEEK}")
    return None if p is None else rd(p)


def check(part, name, value, expected, ok, note=""):
    CHECKS.append({"part": part, "check": name, "value": value, "expected": expected,
                   "status": "OK" if ok else ("INFO" if ok is None else "CHECK"), "note": note})


def show(df, title, n=None):
    display(Markdown(f"**{title}**"))
    display(df if n is None else df.head(n))


def period(v):
    v = pd.to_numeric(v, errors="coerce")
    return pd.Series(np.select([v <= 1, v <= 7, v >= 28], ["daily", "weekly", "monthly"], "other"), index=v.index).where(v.notna(), "not in catalogue")


print(f"run {D}: {OUT}\nweek (catalogue, scripts, conflicts): {WEEK}")

## Load the run

The final file, the two 360 views, and from the week folder the catalogue, the SMS scripts, the conflict pairs and the 270-day purchases (for the favourites).

In [ ]:
_fp = OUT / "final" / "input_flytxt.csv.gz"
if not _fp.exists():
    _fp = OUT / TARGET / f"input_flytxt_{TARGET}.csv.gz"
fly = rd(_fp)
fly.index.name = "LINE"
s360 = pd.read_parquet(OUT / TARGET / f"{TARGET}_scored_360.parquet")
a360 = pd.read_parquet(OUT / TARGET / f"{TARGET}_360.parquet")

cat = table("btl_catalog")
cat.columns = [c.strip().upper() for c in cat.columns]
cat["BTL_NAME"] = cat["BTL_NAME"].str.strip()
for _c in ("PRICE", "GB", "MIN", "VALIDITY"):
    cat[_c] = pd.to_numeric(cat[_c], errors="coerce") if _c in cat.columns else np.nan
for _c in ("TYPE", "SERVICE", "GENEROSITY"):
    cat[_c] = cat[_c].astype(str).str.strip().str.upper() if _c in cat.columns else ""
cat = cat.drop_duplicates("BTL_NAME").set_index("BTL_NAME")

scr = table("BTL_script_english_arabic_ok")
scr.columns = [c.strip().upper() for c in scr.columns]
scr = scr.assign(DESTINATION_OFFER_NAME=scr["DESTINATION_OFFER_NAME"].str.strip()).drop_duplicates("DESTINATION_OFFER_NAME").set_index("DESTINATION_OFFER_NAME")

_cf = table("conflicting_offers_h")
CONFLICTS = set()
if _cf is not None:
    _cf.columns = [c.strip().lower() for c in _cf.columns]
    _cf = _cf[_cf.get("offer_type_a", pd.Series("", index=_cf.index)).astype(str) != "VOICE"]     # as code 5 (F-12)
    CONFLICTS = {frozenset((a.strip(), b.strip())) for a, b in zip(_cf["offer_a"], _cf["offer_b"])}

_h = table("sub_btl_3months")
HIST = set()
if _h is not None:
    _vcol = next(c for c in _h.columns if c.strip().lower() == "access_method_val")
    _pcol = next(c for c in _h.columns if c.strip().upper() == "PRODUCTNAME")
    HIST = set(zip(_h[_vcol].str.strip().str[-8:], _h[_pcol].str.strip()))

# one row per (line, slot) of the final file, with the catalogue facts and, from the 360 view, why it is there
fly["_V"] = fly["msisdn"].str.strip().str[-8:]
_r = s360.assign(_V=s360["ACCESS_METHOD_VAL"].astype(str).str.strip().str[-8:]).drop_duplicates("_V").set_index("_V")
long = pd.concat([pd.DataFrame({"LINE": fly.index, "_V": fly["_V"], "SLOT": i, "ACTION": fly["segment"],
                                "OFFER": fly[f"destination_offer_name_{i}"].str.strip(),
                                "OPPORTUNITY": fly[f"opportunity_name_{i}"].str.strip(),
                                "SCRIPT_EN": fly[f"offer_script_en_{i}"].str.strip(),
                                "SCRIPT_AR": fly[f"offer_script_ar_{i}"].str.strip()}) for i in SLOTS], ignore_index=True)
long["REASON"] = [(_r.at[v, f"REASON_{i}"] if v in _r.index and f"REASON_{i}" in _r.columns else None)
                  for v, i in zip(long["_V"], long["SLOT"])]
long["SELECTED_BY"] = [(_r.at[v, f"SLOT_{i}_SELECTED_BY"] if v in _r.index and f"SLOT_{i}_SELECTED_BY" in _r.columns else None)
                       for v, i in zip(long["_V"], long["SLOT"])]
long["VS_ANCHOR"] = [(_r.at[v, f"SLOT_{i}_VS_ANCHOR"] if v in _r.index and f"SLOT_{i}_VS_ANCHOR" in _r.columns else np.nan)
                     for v, i in zip(long["_V"], long["SLOT"])]
long["SEGMENT"] = long["_V"].map(_r["SEGMENT"]) if "SEGMENT" in _r.columns else None
long["GROUP_KEY"] = long["_V"].map(_r["GROUP_KEY"]) if "GROUP_KEY" in _r.columns else None
EMPTY = long[long["OFFER"].eq("")]
long = long[long["OFFER"].ne("")].join(cat[["TYPE", "SERVICE", "GENEROSITY", "VALIDITY", "PRICE", "GB", "MIN"]], on="OFFER")
long["PERIOD"] = period(long["VALIDITY"])
long["REASON"] = long["REASON"].fillna("-").astype(str)
print(f"final file {_fp.relative_to(ROOT)}: {len(fly):,} rows x {fly.shape[1] - 1} columns | {len(long):,} offers | "
      f"360 scored view {len(s360):,} rows, 360 all target lines {len(a360):,} | catalogue {len(cat):,} offers")

## Part 1 · What was scored

In [ ]:
_in = a360["IN_INPUT_FLYTXT"].astype(bool) if "IN_INPUT_FLYTXT" in a360.columns else pd.Series(True, index=a360.index)
scope = a360.groupby("GROUP_KEY").agg(target_lines=("ACCESS_METHOD_VAL", "size"), in_final_file=("IN_INPUT_FLYTXT", "sum")) \
    if "GROUP_KEY" in a360.columns else pd.DataFrame({"target_lines": [len(a360)], "in_final_file": [int(_in.sum())]})
scope["not_in_file"] = scope["target_lines"] - scope["in_final_file"]
scope["share_in_file"] = (scope["in_final_file"] / scope["target_lines"]).round(4)
show(scope.reset_index(), "1a. Target lines and lines in the final file, per group")
drops = (a360.loc[~_in, "DROP_RULE"].fillna("(no rule given)").value_counts().rename_axis("DROP_RULE").reset_index(name="lines")
         if "DROP_RULE" in a360.columns else pd.DataFrame())
show(drops, "1b. Why the other lines are not in the file (DROP_RULE)")
check(1, "target lines in the final file", f"{int(_in.sum()):,} of {len(a360):,}", "most of them", None,
      f"{len(a360) - int(_in.sum()):,} dropped, see 1b")

## Part 2 · The final file

In [ ]:
_dupl = int(fly["msisdn"].duplicated().sum())
_n = long.groupby("LINE").size().reindex(fly.index, fill_value=0)
per_line = _n.value_counts().sort_index().rename_axis("offers in the line").reset_index(name="lines")
show(pd.DataFrame({"what": ["rows", "distinct MSISDN", "duplicated MSISDN", "columns", "date", "start_date", "end_date"],
                   "value": [len(fly), fly["msisdn"].nunique(), _dupl, fly.shape[1] - 1, ", ".join(sorted(fly["date"].unique())),
                             ", ".join(sorted(fly["start_date"].unique())), ", ".join(sorted(fly["end_date"].unique())) or "(empty)"]}),
     "2a. The file")
show(per_line, "2b. Offers per line")
show(EMPTY.groupby("SLOT").size().reindex(list(SLOTS), fill_value=0).rename_axis("slot").reset_index(name="empty"),
     "2c. Empty slots, per slot")
show(fly["segment"].value_counts().rename_axis("segment (= action)").reset_index(name="lines"), "2d. The segment column (the action sent)")
show(fly["segment_btl_taker"].value_counts().rename_axis("segment_btl_taker").reset_index(name="lines"), "2e. BTL takers (270 days)")
check(2, "one row per MSISDN", _dupl, 0, _dupl == 0)
check(2, "38 columns", fly.shape[1] - 1, 38, fly.shape[1] - 1 == 38)
check(2, "lines with 6 offers", f"{int((_n == 6).sum()):,} of {len(fly):,}", "all", bool((_n == 6).all()))

## Part 3 · Every offer: catalogue, scripts, opportunity name

In [ ]:
_bad_cat = long[long["PRICE"].isna()]
_no_en, _no_ar = long[long["SCRIPT_EN"].eq("")], long[long["SCRIPT_AR"].eq("")]
_exp_en = long["OFFER"].map(scr["ENGLISH SCRIPT"].str.strip()) if "ENGLISH SCRIPT" in scr.columns else long["SCRIPT_EN"]
_exp_ar = long["OFFER"].map(scr["ARABIC SCRIPT"].str.strip()) if "ARABIC SCRIPT" in scr.columns else long["SCRIPT_AR"]
_diff_en, _diff_ar = long[long["SCRIPT_EN"].ne(_exp_en.fillna(""))], long[long["SCRIPT_AR"].ne(_exp_ar.fillna(""))]
_opp = long[long["OPPORTUNITY"].ne("nbo_LFM_" + long["OFFER"])]
_opp_empty = EMPTY[EMPTY["OPPORTUNITY"].ne("") | EMPTY["SCRIPT_EN"].ne("") | EMPTY["SCRIPT_AR"].ne("")]
show(pd.DataFrame({"check": ["offer not in the catalogue", "no English script", "no Arabic script",
                             "English script differs from the script table", "Arabic script differs from the script table",
                             "opportunity name is not nbo_LFM_<offer>", "empty slot with an opportunity name or a script"],
                   "offers": [len(_bad_cat), len(_no_en), len(_no_ar), len(_diff_en), len(_diff_ar), len(_opp), len(_opp_empty)]}),
     "3a. Every offer of the file (expected 0 everywhere)")
if len(_bad_cat):
    show(_bad_cat["OFFER"].value_counts().rename_axis("offer").reset_index(name="times"), "3b. Offers not in the catalogue", EXAMPLES)
for _name, _df in (("offer in the catalogue", _bad_cat), ("English script", _no_en), ("Arabic script", _no_ar),
                   ("scripts = the script table", pd.concat([_diff_en, _diff_ar])), ("opportunity name", _opp)):
    check(3, _name, len(_df), 0, len(_df) == 0)

## Part 4 · The code-5 rules: F-11, F-09, F-12

In [ ]:
_k = long[["LINE", "SLOT", "OFFER", "TYPE", "PRICE", "VALIDITY", "SERVICE", "GB", "MIN", "REASON"]]
pairs = _k.merge(_k, on="LINE", suffixes=("_A", "_B"))
pairs = pairs[pairs["SLOT_A"] < pairs["SLOT_B"]]
f11 = pairs[pairs["OFFER_A"] == pairs["OFFER_B"]]
f09 = pairs[(pairs["OFFER_A"] != pairs["OFFER_B"]) & (pairs["TYPE_A"] == pairs["TYPE_B"]) & (pairs["PRICE_A"] == pairs["PRICE_B"])]
f12 = pairs[[frozenset((a, b)) in CONFLICTS for a, b in zip(pairs["OFFER_A"], pairs["OFFER_B"])]]
show(pd.DataFrame({"rule": ["F-11 the same offer twice in a line", "F-09 two offers of the same TYPE and PRICE",
                            "F-12 a conflict pair (conflicting_offers_h) in a line"],
                   "lines": [f11["LINE"].nunique(), f09["LINE"].nunique(), f12["LINE"].nunique()],
                   "pairs": [len(f11), len(f09), len(f12)]}), "4a. The code-5 rules (expected 0)")
for _t, _df in (("F-11", f11), ("F-09", f09), ("F-12", f12)):
    if len(_df):
        show(_df[["LINE", "SLOT_A", "OFFER_A", "SLOT_B", "OFFER_B"]], f"4b. {_t} examples", EXAMPLES)
    check(4, _t, int(_df["LINE"].nunique()), 0, len(_df) == 0)

## Part 5 · Daily, weekly, monthly

Slots 1, 2, 5 hold the 30-day offers and 3, 4, 6 the 7-day offers (F-02 / F-03). A favourite or a forced bundle fills an empty slot 5 / 6 with whatever validity it has.

In [ ]:
long["SLOT_KIND"] = np.where(long["SLOT"].isin(MONTHLY_SLOTS), "monthly slot (1, 2, 5)", "weekly slot (3, 4, 6)")
show(pd.crosstab(long["SLOT"], long["PERIOD"], margins=True, margins_name="total"), "5a. Slot x validity of the offer")
show(long["PERIOD"].value_counts().rename_axis("validity").reset_index(name="offers")
     .assign(share=lambda d: (d["offers"] / d["offers"].sum()).round(4)), "5b. Daily / weekly / monthly offers")
_mix = long.pivot_table(index="LINE", columns="PERIOD", values="OFFER", aggfunc="size", fill_value=0)
for _p in ("monthly", "weekly", "daily"):
    if _p not in _mix.columns:
        _mix[_p] = 0
_pattern = (_mix["monthly"].astype(str) + " monthly + " + _mix["weekly"].astype(str) + " weekly"
            + np.where(_mix["daily"] > 0, " + " + _mix["daily"].astype(str) + " daily", ""))
show(_pattern.value_counts().rename_axis("mix in the line").reset_index(name="lines"), "5c. The mix per line")
_wrong = long[((long["SLOT"].isin(MONTHLY_SLOTS)) & long["PERIOD"].ne("monthly")) | ((long["SLOT"].isin(WEEKLY_SLOTS)) & long["PERIOD"].ne("weekly"))]
show(_wrong.groupby(["SLOT_KIND", "PERIOD", "REASON"]).size().rename("offers").reset_index(),
     "5d. Offers whose validity does not match their slot, by reason (a favourite or a forced bundle may do this)")
_wrong_rec = _wrong[_wrong["REASON"].str.startswith("recommended")]
check(5, "recommended offers in a slot of another validity", len(_wrong_rec), 0, len(_wrong_rec) == 0)
check(5, "lines with 3 monthly + 3 weekly", f"{int(((_mix['monthly'] == 3) & (_mix['weekly'] == 3)).sum()):,} of {len(fly):,}",
      "most of them", None, "the rest: favourites / forced bundles of another validity, see 5d")

## Part 6 · Why each offer is there, and the favourites

In [ ]:
_reason = long["REASON"].str.replace(r"forced F-08 .*", "forced F-08 (bundle)", regex=True)
show(_reason.value_counts().rename_axis("why the offer is there").reset_index(name="offers")
     .assign(share=lambda d: (d["offers"] / d["offers"].sum()).round(4)), "6a. Why each offer is there")
show(pd.crosstab(long["SLOT"], _reason), "6b. Per slot")
if long["SELECTED_BY"].notna().any():
    show(long["SELECTED_BY"].fillna("-").value_counts().rename_axis("selected by (rule / pass)").reset_index(name="offers"),
         "6c. Which rule selected the recommended offers")
fav = long[long["REASON"].eq("favourite")]
_real = pd.Series([(v, o) in HIST for v, o in zip(fav["_V"], fav["OFFER"])], index=fav.index)
show(pd.DataFrame({"what": ["lines with at least 1 favourite", "favourite offers", "favourites in slot 5", "favourites in slot 6",
                            "favourites really bought in the 270 days (sub_btl_3months)", "favourites NOT found in the history"],
                   "value": [fav["LINE"].nunique(), len(fav), int((fav["SLOT"] == 5).sum()), int((fav["SLOT"] == 6).sum()),
                             int(_real.sum()), int((~_real).sum())]}), "6d. Favourites")
show(fav.groupby("PERIOD").size().rename_axis("validity").reset_index(name="favourites"), "6e. Favourites by validity")
check(6, "favourites found in the customer's purchase history", f"{int(_real.sum()):,} of {len(fav):,}", "all", bool(_real.all()))
check(6, "lines with a favourite", f"{fav['LINE'].nunique():,} of {len(fly):,}", "-", None)

## Part 7 · Same price, different allocation, and dominated offers

- **Same price, different allocation:** two offers of one line with the same price and validity but other GB or minutes, e.g. 5 R for 15 GB and 5 R for 20 GB. Code 5 (F-09) only clears two offers of the same **TYPE** and price, so this can pass.
- **Dominated:** in one line, same validity and service, one offer costs the same or more and gives less or equal GB **and** minutes. The customer would never choose it: it wastes a slot.

In [ ]:
same = pairs[(pairs["OFFER_A"] != pairs["OFFER_B"]) & (pairs["PRICE_A"] == pairs["PRICE_B"])].copy()
same["SAME_VALIDITY"] = same["VALIDITY_A"] == same["VALIDITY_B"]
same["ALLOCATION_DIFFERS"] = (same["GB_A"].fillna(-1) != same["GB_B"].fillna(-1)) | (same["MIN_A"].fillna(-1) != same["MIN_B"].fillna(-1))
alloc = same[same["SAME_VALIDITY"] & same["ALLOCATION_DIFFERS"]]
show(pd.DataFrame({"case": ["two offers with the same price in a line (any validity)",
                            "same price AND same validity, different GB / minutes",
                            "of which the same service (e.g. two DATA offers)"],
                   "lines": [same["LINE"].nunique(), alloc["LINE"].nunique(), alloc[alloc["SERVICE_A"] == alloc["SERVICE_B"]]["LINE"].nunique()],
                   "pairs": [len(same), len(alloc), int((alloc["SERVICE_A"] == alloc["SERVICE_B"]).sum())]}),
     "7a. Same price, different allocation in one line")
_top = (alloc.assign(PAIR=lambda d: d["OFFER_A"] + "  |  " + d["OFFER_B"])
        .groupby(["PAIR", "PRICE_A", "VALIDITY_A"]).agg(lines=("LINE", "nunique"), GB_A=("GB_A", "first"), GB_B=("GB_B", "first"),
                                                       MIN_A=("MIN_A", "first"), MIN_B=("MIN_B", "first"),
                                                       reasons=("REASON_B", lambda s: ", ".join(sorted(set(s)))))
        .sort_values("lines", ascending=False).reset_index())
show(_top, "7b. The pairs, most frequent first (same price, same validity, different allocation)", EXAMPLES)

# dominated: same validity and service, B costs the same or more and gives less or equal GB AND minutes (one strictly)
_d = pairs[(pairs["OFFER_A"] != pairs["OFFER_B"]) & (pairs["VALIDITY_A"] == pairs["VALIDITY_B"]) & (pairs["SERVICE_A"] == pairs["SERVICE_B"])].copy()
_ga, _gb, _ma, _mb = (_d[c].fillna(0) for c in ("GB_A", "GB_B", "MIN_A", "MIN_B"))
_a_worse = (_d["PRICE_A"] >= _d["PRICE_B"]) & (_ga <= _gb) & (_ma <= _mb) & ((_d["PRICE_A"] > _d["PRICE_B"]) | (_ga < _gb) | (_ma < _mb))
_b_worse = (_d["PRICE_B"] >= _d["PRICE_A"]) & (_gb <= _ga) & (_mb <= _ma) & ((_d["PRICE_B"] > _d["PRICE_A"]) | (_gb < _ga) | (_mb < _ma))
dom = _d[_a_worse | _b_worse].assign(WORSE=np.where(_a_worse[_a_worse | _b_worse], "A", "B"))
dom["WORSE_OFFER"] = np.where(dom["WORSE"] == "A", dom["OFFER_A"], dom["OFFER_B"])
dom["BETTER_OFFER"] = np.where(dom["WORSE"] == "A", dom["OFFER_B"], dom["OFFER_A"])
dom["WORSE_REASON"] = np.where(dom["WORSE"] == "A", dom["REASON_A"], dom["REASON_B"])
show(pd.DataFrame({"case": ["a line holds an offer that is dominated by another offer of the same line"],
                   "lines": [dom["LINE"].nunique()], "pairs": [len(dom)]}),
     "7c. Dominated offers (same validity and service: the same price or more for less or equal GB and minutes)")
show(dom.groupby(["WORSE_OFFER", "BETTER_OFFER"]).agg(lines=("LINE", "nunique"), worse_is=("WORSE_REASON", lambda s: ", ".join(sorted(set(s)))))
     .sort_values("lines", ascending=False).reset_index(), "7d. Dominated pairs, most frequent first", EXAMPLES)
show(dom["WORSE_REASON"].value_counts().rename_axis("the dominated offer is there because").reset_index(name="pairs"),
     "7e. Where the dominated offers come from")
check(7, "lines with 2 offers of the same price and validity but a different allocation", f"{alloc['LINE'].nunique():,}", 0,
      alloc.empty, "see 7b: F-09 only clears the same TYPE and price")
check(7, "lines with a dominated offer", f"{dom['LINE'].nunique():,}", 0, dom.empty, "see 7d / 7e")

## Part 8 · Price order and price against the usual spend

In [ ]:
_rec = long[long["REASON"].str.startswith("recommended")]
_order = []
for _slots, _name in ((MONTHLY_SLOTS, "monthly"), (WEEKLY_SLOTS, "weekly")):
    _p = _rec[_rec["SLOT"].isin(_slots)].pivot_table(index="LINE", columns="SLOT", values="PRICE", aggfunc="first")
    _p = _p.reindex(columns=list(_slots))
    _bad = ((_p[_slots[0]] < _p[_slots[1]]) | (_p[_slots[1]] < _p[_slots[2]])).sum()
    _order.append({"slots": f"{_name} {_slots}", "lines with 3 recommended": int(_p.notna().all(axis=1).sum()),
                   "lines not in price order (highest first)": int(_bad)})
show(pd.DataFrame(_order), "8a. Price order of the recommended offers (F-02 / F-03: highest price in the first slot)")
_va = pd.to_numeric(long["VS_ANCHOR"], errors="coerce")
_band = pd.cut(_va, [-np.inf, 0.5, 0.8, 1.0, 1.5, 2.0, 3.0, np.inf], right=False,
               labels=["< 0.5", "0.5-0.8", "0.8-1.0", "1.0-1.5", "1.5-2.0", "2.0-3.0", ">= 3.0"])
show(pd.crosstab(_band.cat.add_categories("no anchor").fillna("no anchor"), long["REASON"].str.replace(r"forced F-08 .*", "forced F-08", regex=True),
                 margins=True, margins_name="total"),
     "8b. Price / the customer's usual spend (anchor), by reason: < 1.0 = below the usual spend (cannibalization risk)")
_below = long[_va < 1.0]
show(_below.groupby(["REASON", "PERIOD"]).size().rename("offers").reset_index(), "8c. Offers below the usual spend")
show(long.groupby("PERIOD")["PRICE"].describe().round(2), "8d. Prices per validity")
_cols = [c for c in ("CHK_OFFERS_BELOW_ANCHOR", "CHK_FAVOURITE_BELOW_ANCHOR", "CHK_FALLBACK_PASS_BELOW_ANCHOR", "CHK_TOPUP_BELOW_ANCHOR") if c in s360.columns]
show(pd.DataFrame({c: [int((pd.to_numeric(s360[c], errors="coerce") > 0).sum())] for c in _cols}, index=["lines"]),
     "8e. The flags of the 360 view (lines with at least one)")
_rec_below = _rec[(_va.reindex(_rec.index) < 1.0) & _rec["SELECTED_BY"].fillna("").astype(str).str.startswith("challenger")]
check(8, "recommended offers not in price order", int(sum(r["lines not in price order (highest first)"] for r in _order)), 0,
      all(r["lines not in price order (highest first)"] == 0 for r in _order))
check(8, "lines with an offer below the usual spend", f"{_below['LINE'].nunique():,} of {len(fly):,}", "few",
      None, "favourites / fallback passes / top-up may be below it: see 8c")
check(8, "challenger-band offers below the usual spend", len(_rec_below), 0, _rec_below.empty)

## Part 9 · The actions and their rules

In [ ]:
_l = s360.copy()
_act = _l["ACTION_SENT"].astype(str) if "ACTION_SENT" in _l.columns else _l["ACTION_AGENT"].astype(str)
show(pd.crosstab(_l["SEGMENT"], _act, margins=True, margins_name="total"), "9a. Segment x action sent")
if "DECIDED_BY" in _l.columns:
    show(_l["DECIDED_BY"].value_counts().rename_axis("decided by").reset_index(name="lines"), "9b. Who decided the action")
_h06 = _l[_act.isin(UPSELL_ACTIONS) & _l["SEGMENT"].isin(AT_RISK_SEGMENTS)].assign(ACTION=_act)
_h06["EXEMPT"] = [H06_EXEMPT.get(s) == a for s, a in zip(_h06["SEGMENT"], _h06["ACTION"])]
show(_h06.groupby(["SEGMENT", "ACTION", "EXEMPT"]).size().rename("lines").reset_index(),
     "9c. Upsell to an at-risk segment (H-06). EXEMPT = the segment's own production action (Onboarding at risk = UPSELL)")
_churn = pd.to_numeric(_l.get("PROBA_CHURN"), errors="coerce")
_up60 = int((_act.isin(UPSELL_ACTIONS) & (_churn >= 0.6)).sum())
_nurs = int(_l["SEGMENT"].astype(str).str.lower().str.startswith("nursering").sum())
_fam = long[long["GROUP_KEY"].eq("t2c") & long["ACTION"].isin(ACTION_FAMILIES)].copy()
_fam["IN_FAMILY"] = [s in ACTION_FAMILIES[a] for s, a in zip(_fam["SERVICE"], _fam["ACTION"])]
show(_fam.groupby(["REASON", "IN_FAMILY"]).size().unstack(fill_value=0), "9d. Group a: is the offer's service one of the action's families (BR-66)?")
_t2f = long[long["GROUP_KEY"].eq("t2f")]
_t2f_bad = _t2f[_t2f["REASON"].str.startswith("recommended") & (~_t2f["SERVICE"].isin(["DATA", "COMBO"]) | _t2f["GENEROSITY"].ne("L2"))]
_out_rec = _fam[~_fam["IN_FAMILY"] & _fam["REASON"].str.startswith("recommended")]
check(9, "upsell to an at-risk segment, not exempt (H-06)", int((~_h06["EXEMPT"]).sum()), 0, bool(_h06["EXEMPT"].all()),
      "see 9c" + (f"; {int(_h06['EXEMPT'].sum()):,} exempt lines (Onboarding at risk = UPSELL in production)" if _h06["EXEMPT"].any() else ""))
check(9, "upsell with churn >= 0.60", _up60, "few", None, "information (plan Q9)")
check(9, "Nursering lines in the file", _nurs, 0, _nurs == 0)
check(9, "group a: recommended offers outside the action's families", len(_out_rec), 0, _out_rec.empty, "see 9d")
if len(_t2f):
    check(9, "group b: recommended offers outside L2 DATA / COMBO", len(_t2f_bad), 0, _t2f_bad.empty)

## Part 10 · The 360 view against the file, the most recommended offers

In [ ]:
_c = [c for c in ("CHK_NOT_IN_FLYTXT", "CHK_SLOTS_DIFFER_FROM_FLYTXT") if c in s360.columns]
_v = {c: int(s360[c].astype(bool).sum()) for c in _c}
_extra = len(set(fly["_V"]) - set(s360["ACCESS_METHOD_VAL"].astype(str).str[-8:]))
show(pd.DataFrame({"check": [*_c, "final-file lines not in the scored 360 view"], "lines": [*_v.values(), _extra]}),
     "10a. The 360 view against the final file (expected 0)")
for _k, _x in _v.items():
    check(10, _k, _x, 0, _x == 0)
check(10, "final-file lines not in the 360 view", _extra, 0, _extra == 0)
top = (long.groupby(["OFFER", "PERIOD", "SERVICE", "GENEROSITY", "PRICE", "GB", "MIN"], dropna=False)
       .agg(times=("LINE", "size"), lines=("LINE", "nunique"), as_favourite=("REASON", lambda s: int((s == "favourite").sum())))
       .sort_values("times", ascending=False).reset_index())
top["share"] = (top["times"] / top["times"].sum()).round(4)
show(top, "10b. The most recommended offers", 25)
show(long.groupby(["PERIOD", "SERVICE"]).size().unstack(fill_value=0), "10c. Offers per validity and service")

## Part 11 · Scorecard

In [ ]:
card = pd.DataFrame(CHECKS)
show(card, "11. Scorecard: every check (CHECK = look at the table of that part; INFO = a figure, no rule)")
print(f"{int((card['status'] == 'OK').sum())} OK, {int((card['status'] == 'CHECK').sum())} CHECK, "
      f"{int((card['status'] == 'INFO').sum())} INFO")